# Module 2: Deploy

> Part of the **Modular Workshops** series. Standalone, ~15 min.

Moving the agent to production requires **production-ready deployment infrastructure** — a managed runtime for the agent. **LangSmith Deployments** gives 30+ endpoints, persistence, HITL, and Studio out of the box.

This module ships the deep agent from Module 1 to LangSmith with the `langgraph` CLI.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

from utils.workshop import scoped, workshop_user

print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:   ", bool(os.environ.get("TAVILY_API_KEY")))
print("Workshop user:        ", workshop_user())


---
# Part 1. Deploy — Ship the Agent

We deploy the agent in `agents/deep_agent/` to **LangSmith Deployments** with the `langgraph` CLI. Because `agents/deep_agent/agent.py` imports `model` from `utils.models`, whatever model configuration lives there ships with the image automatically — no extra flags, no separate config.


## 1.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`.

`dependencies: ["."]` tells the CLI to install this project — `pyproject.toml` at the config root — into the image.


In [ ]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


In [ ]:
# langgraph.json -- the deploy configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives in `agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 1.2 Local development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith (beta)
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 1.3 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and checks the config — without building Docker or uploading anything. Use it to catch config and import errors before deploying.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


## 1.4 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> The image we're shipping picks up its model configuration from `utils/models.py` — no extra deploy flags needed.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

> Everything in the project directory becomes part of the Docker build context. Add a `.dockerignore` to keep secrets out of the image — `langgraph deploy` reads `.env` from your local filesystem and uploads the values as deployment secrets, so the image itself never needs the file. Being in `.gitignore` does not exclude it.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type dedicated` — always-on deployment (default is `serverless`; orgs on previous pricing use `dev`/`prod` until Oct 1, 2026)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
# Scoped per attendee so concurrent deploys don't target the same deployment.
deployment_name = scoped("modular-workshops-deep-agent")
print("Deploying as:", deployment_name)

!cd "{project_root}" && langgraph deploy --name {deployment_name} --no-input


## 1.5 What you get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |


## Recap

| What | How |
|---|---|
| Deployable graph config | `langgraph.json` at repo root |
| Agent identity + skills | `AGENTS.md`, `skills/` |
| Validate before shipping | `langgraph validate` |
| Ship it | `langgraph deploy --name <your-deployment-name>` |

**The production pattern:** one config file + one deploy command. The agent now lives behind a managed server, with 30+ endpoints available to call it.

**Next:** Module 3 — LangSmith (prompt engineering, tracing, querying traces, offline + online evals, annotation queues).
